# B3 · Anima (Cosmos-Predict2 DiT + LLM adapter) 학습 동작 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/train_verify/B3_anima.ipynb)

원본: diffusers `CosmosTransformer3DModel` (`models/transformers/transformer_cosmos.py`)
+ `AnimaTextConditioner` (`models/condition_embedders/condition_embedder_anima.py`).
diffusers `v0.41.0` (commit `086bf95`)에서 처음 PyPI에 들어왔고, 두 파일은 main `af4271c`와 같습니다.
실제 체크포인트: `circlestone-labs/Anima-Base-v1.0-Diffusers` (`073c3a9`).
내 구현: `miniarch/models/anima.py`. 가중치 이름은 diffusers와 같습니다 (`text_conditioner.*`, `transformer.*`).

Anima = Cosmos-Predict2-2B T2I DiT를 그대로 쓰고, 텍스트 쪽만 바꾼 모델입니다.
- LLM adapter: T5 토큰 임베딩을 질의로 삼아 Qwen3-0.6B hidden을 cross-attn으로 읽습니다 (6블록, q/k RMSNorm + RoPE).
  결과를 RMSNorm하고 T5 padding을 0으로 만든 뒤 512 토큰까지 0으로 채웁니다.
- Cosmos DiT: padding mask 채널 concat, (1,2,2) 3D patch, 3D RoPE (head 채널을 t/h/w로 나누고 h·w에 NTK 4배),
  adaLN-LoRA 변조 (블록별 저랭크 C→256→3C + 모든 블록이 공유하는 temb), self-attn → 텍스트 cross-attn → GELU FFN, 28블록.

비교 방식: 파이프라인의 한 step처럼 adapter → DiT를 이어 붙인 래퍼를 원본으로 두고,
adapter 출력, adapter 블록, RoPE, patch, 시간 임베딩, 블록별 attn1/attn2/ff/블록 출력, 최종 출력을 층별로 비교합니다.

축소 config
- DiT: hidden 56 (head 2 × 28), 2블록, adaLN-LoRA 8. head_dim 28 → t/h/w 채널 12/8/8로 서로 달라서 축이 뒤바뀌면 드러납니다.
  latent 12×10 (patch 후 6×5, 가로세로가 달라 h/w 뒤바뀜을 잡음), rope_scale (1, 4, 4)는 Anima와 같습니다.
- padding mask는 latent 2배 크기의 무작위 0/1 → nearest resize 경로를 지나갑니다 (파이프라인은 전부 0).
- adapter: model_dim 32 ≠ target_dim 24 → `in_proj` Linear 경로, 2블록, head 4. Qwen 9토큰 / T5 7토큰,
  배치 1번은 Qwen 뒤 3칸, T5 뒤 2칸이 padding. min_sequence_length 12 → 0 행 5개를 붙여 DiT가 mask 없이 읽습니다.
- 자체 점검에서 Cosmos-Predict2 비디오 경로(5D timestep, fps, learnable 위치, cross-attn mask,
  adapter LayerNorm / self-attn 없음 / in_proj 없음)도 원본과 한 번 더 비교합니다.

순서: 0. 환경 준비 → 1. 기울기 == 원본 → 2. 학습 궤적 == 원본 → 3. 배치 하나 외우기 → 4. float64 수치 미분 → 5. 라우터 bias 업데이트 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
if not os.environ.get('MINIARCH_SKIP_PIP'):   # 전체 실행 노트북이 한 번에 설치
    !pip -q install -U 'diffusers==0.41.0'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("B3")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 학습 동작 확인

forward 가 원본과 같아도 학습이 되는지는 따로 확인해야 합니다.
- **기울기 == 원본**: 같은 가중치·같은 손실로 원본과 내 구현을 backward 해서 파라미터 기울기를 비교합니다 (파라미터별 상대 오차 ≤ 1e-3).
  원본이 추론 전용 코드(no_grad)면 건너뜁니다.
- **학습 궤적 == 원본**: 같은 가중치·데이터·optimizer(SGD momentum 0.9, 30 step)로 원본과 내 구현을 나란히 학습해
  step 별 손실과 최종 가중치를 비교합니다 (상대차 ≤ 1e-3). Adam 은 아주 작은 잡음 차이를 키워서 궤적 비교에 쓰지 않습니다.
- **배치 하나 외우기**: 고정 배치 하나로 300 step 학습해 손실이 처음의 20% 이하로 내려가는지 봅니다
  (LM은 다음 토큰 CE, 그 외는 가중치를 살짝 바꾼 같은 구조(teacher)의 출력에 대한 MSE — 항상 도달 가능한 목표).
  NaN/inf 가 없어야 하고, 기울기를 한 번도 받지 못한 파라미터를 보여줍니다 (MoE 에서 안 뽑힌 expert 는 정상).

In [ ]:
import importlib, traceback
from miniarch.verify import training as TR
mod = importlib.import_module(S.CASE.mine_module)
V.set_deterministic(0)
ref, inputs = S.ref()
V.perturb_params(ref)
pair = mod.verify_pair(ref)
INF_ONLY = False   # 원본이 추론 전용이면 True → 아래 수치 미분으로 backward 확인
try:
    g = TR.grad_equivalence(ref, pair, inputs, dev)
    if g.note == TR.INFERENCE_ONLY:
        INF_ONLY = True
        print("⏭️", g.note); report("기울기 == 원본 (원본 추론 전용 → 수치 미분으로 대체)", True)
    else:
        print(f"비교한 파라미터 {g.compared}개, 최대 상대 오차 {g.max_rel:.2e} ({g.worst})", g.note)
        if g.skipped_ref:
            print(f"원본에서 기울기가 안 흐르는 파라미터 {len(g.skipped_ref)}개 (비교 제외):", g.skipped_ref[:6])
        report("기울기 == 원본", g.ok, g.max_rel)
except Exception:
    traceback.print_exc(limit=4); report("기울기 == 원본", False)

In [ ]:
try:
    t = TR.train_parity(ref, pair, inputs, dev)
    if t.note == TR.INFERENCE_ONLY:
        print("⏭️", t.note); report("학습 궤적 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"{t.steps} step  손실 최대 상대차 {t.max_loss_rel:.2e}  최종 가중치 최대 상대차 {t.max_param_rel:.2e} ({t.worst_param})", t.note)
        if t.frozen:
            print(f"  원본에서 기울기가 안 흘러 양쪽 다 얼린 파라미터 {len(t.frozen)}개:", t.frozen[:6])
        print("  원본:", " ".join(f"{v:.4g}" for v in t.loss_ref[::6]))
        print("  내것:", " ".join(f"{v:.4g}" for v in t.loss_mine[::6]))
        report("학습 궤적 == 원본 (SGD 30 step)", t.ok, max(t.max_loss_rel, t.max_param_rel))
except Exception:
    traceback.print_exc(limit=4); report("학습 궤적 == 원본", False)

In [ ]:
try:
    o = TR.overfit_batch(pair["mine"], pair, inputs, dev)
    print(f"{o.kind}: {o.loss_start:.4f} → {o.loss_end:.4f}  (유한={o.finite})")
    print("  손실:", " ".join(f"{v:.3g}" for v in o.curve[::30]))
    if o.no_grad_params:
        print(f"  기울기를 못 받은 파라미터 {len(o.no_grad_params)}개:", o.no_grad_params[:8])
    report("배치 하나 외우기 (" + o.kind + ")", o.ok, o.loss_end / max(o.loss_start, 1e-12))
except Exception:
    traceback.print_exc(limit=4); report("배치 하나 외우기", False)

### 원본과 비교할 수 없는 부분

- **수치 미분**: 무작위 방향으로 가중치를 ±ε 움직여 잰 손실 변화가 autograd 기울기와 같은지 봅니다.
  원본이 추론 전용이라 위의 "기울기 == 원본"을 못 한 모델에서만 돌립니다 (그 외에는 원본 비교가 더 엄밀해서 건너뜀).
- **라우터 bias 업데이트 (aux-loss-free)**: 원본 추론 코드에는 없는 학습 전용 규칙입니다. 많이 뽑힌 expert 의 bias 는 내려가고
  적게 뽑힌 쪽은 올라가는지, bias 가 optimizer 에서 빠져 있는지, bias 가 실제로 선택을 바꾸는지 봅니다. 라우터 bias 가 없는 모델은 건너뜁니다.

In [ ]:
if INF_ONLY:
    try:
        f = TR.directional_grad_check(pair["mine"], pair, inputs, dev)
        print(f"방향 {len(f.rel_errors)}개 상대 오차 {[f'{e:.1e}' for e in f.rel_errors]}  (ε={f.eps}, {f.note})")
        report("수치 미분 == autograd", f.ok, max(f.rel_errors) if f.rel_errors else float("nan"))
    except Exception:
        traceback.print_exc(limit=4); report("수치 미분 == autograd", False)
else:
    report("수치 미분 (원본 기울기 비교로 대체 → 건너뜀)", True)
try:
    b = TR.router_bias_check(pair["mine"], pair, inputs, dev)
    print("\n".join(b.details))
    report("라우터 bias 업데이트" if b.n_routers else "라우터 bias 업데이트 (라우터 bias 없음 → 건너뜀)", b.ok)
except Exception:
    traceback.print_exc(limit=4); report("라우터 bias 업데이트", False)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/train_verify/B3_anima.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/train_B3_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")